# Laboratório de metaheurísticas: Mochila 0-1

Este notebook explica as **peças** de `metaheuristics/utils/` e roda a linha de base (SA, ILS e VNS puros). As misturas ficam por sua conta, na última seção.

Rode de cima para baixo, com o kernel do ambiente do Poetry (`poetry install` na raiz do repositório). O catálogo completo das peças está em [`utils/README.md`](utils/README.md).

In [ ]:
import random

from metaheuristics.iterated_local_search import iterated_local_search_mochila
from metaheuristics.simulated_annealing import simulated_annealing_mochila
from metaheuristics.utils import binaria
from metaheuristics.utils import ordem
from metaheuristics.utils.aceitacao import aceita_melhora
from metaheuristics.utils.aceitacao import metropolis
from metaheuristics.utils.aceitacao import temperaturas
from metaheuristics.utils.debug import Depurador
from metaheuristics.utils.experimento import resumir
from metaheuristics.utils.experimento import rodar_sementes
from metaheuristics.utils.experimento import tabela
from metaheuristics.utils.instancia import Item
from metaheuristics.utils.instancia import gerar_instancia_aleatoria
from metaheuristics.utils.instancia import programacao_dinamica
from metaheuristics.utils.plot import plotar_comparacao
from metaheuristics.utils.plot import plotar_historico
from metaheuristics.variable_neighborhood_search import variable_neighborhood_search_mochila

## 1. A instância

A instância são os dados de entrada: os itens (peso e valor) e a capacidade. Ela não muda enquanto o algoritmo roda.

- **Comum:** peso e valor sorteados separadamente. Há itens claramente bons e claramente ruins, e por isso quase qualquer método acerta.
- **Correlacionada:** o valor acompanha o peso (`valor = peso + 2`, a instância "fortemente correlacionada" de Pisinger). Nenhum item é óbvio, e o difícil passa a ser **encaixar pesos** para encher a mochila. É nela que as combinações se diferenciam.

O ótimo de cada uma vem da programação dinâmica e serve para calcular o gap.

In [ ]:
N_ITENS = 50

random.seed(42)
itens_comum = gerar_instancia_aleatoria(N_ITENS, 20, 100)
random.seed(42)
itens = gerar_instancia_aleatoria(N_ITENS, 20, 100, correlacionada=True)

capacidade_comum = sum(item.peso for item in itens_comum) // 2
capacidade = sum(item.peso for item in itens) // 2
otimo_comum, _ = programacao_dinamica(itens_comum, capacidade_comum)
otimo, _ = programacao_dinamica(itens, capacidade)

print("comum:          ", itens_comum[:5], "... C =", capacidade_comum, "· ótimo", otimo_comum)
print("correlacionada: ", itens[:5], "... C =", capacidade, "· ótimo", otimo)

## 2. As duas representações

A mesma solução (quais itens estão na mochila) pode ser escrita de dois jeitos:

| | Binária | Por ordem |
|---|---|---|
| Solução | `[1, 0, 0, 1, 0, 0, 0]`: um bit por item | `[0, 3, 1, 2, ...]`: a ordem em que se tenta colocar os itens |
| Análoga a | "por arestas" no caixeiro viajante | "por vértices" no caixeiro viajante |
| Viabilidade | pode passar da capacidade: precisa de reparo ou valor 0 | sempre viável: o decodificador só coloca o que cabe |
| Redundância | cada solução é um vetor só | ordens diferentes dão a mesma solução |

O exemplo abaixo usa a instância do README do guloso (C = 9). Os índices começam em 0, então o "item 1" do README é o índice 0.

In [ ]:
exemplo = [Item(p, v) for p, v in zip([4, 3, 7, 5, 3, 2, 3], [12, 8, 17, 11, 6, 2, 2])]

# a mesma solução nas duas representações: itens 1 e 4, valor 23
print(binaria.calcular_valor_peso([1, 0, 0, 1, 0, 0, 0], exemplo, 9))
print(ordem.decodificar([0, 3, 1, 2, 4, 5, 6], exemplo, 9), ordem.avaliar([0, 3, 1, 2, 4, 5, 6], exemplo, 9))

# redundância: outra ordem, a mesma solução
print(ordem.decodificar([3, 0, 6, 5, 4, 2, 1], exemplo, 9))

# na binária, um vetor pode passar da capacidade (valor 0); a perturbação repara
print(binaria.calcular_valor_peso([1, 1, 1, 0, 0, 0, 0], exemplo, 9))
random.seed(0)
print(binaria.perturbar([1, 0, 0, 1, 0, 0, 0], exemplo, 9, k=3))

## 3. As peças

Peças com o mesmo **encaixe** (o que recebem e o que devolvem) e a mesma representação são trocáveis. As duas representações usam os mesmos nomes: trocar de representação é trocar o prefixo `binaria.` por `ordem.`. Aceitação, histórico, gráficos e debug servem para as duas representações.

| Encaixe | Binária | Por ordem |
|---|---|---|
| avaliar `(solucao, itens, C) → valor` | `binaria.avaliar` | `ordem.avaliar` |
| construtiva `(itens, C) → solucao` | `binaria.construtiva_aleatoria`, `binaria.construtiva_gulosa` | `ordem.construtiva_aleatoria`, `ordem.construtiva_gulosa` |
| movimento `(solucao, itens, C, k) → solucao` | `binaria.gerar_vizinho` | `ordem.gerar_vizinho` |
| perturbação `(solucao, itens, C, k) → solucao` | `binaria.perturbar` | `ordem.perturbar` |
| busca local `(solucao, itens, C) → (solucao, avaliações)` | `binaria.busca_local_melhor_melhora`, `binaria.busca_local_primeira_melhora` | `ordem.busca_local_primeira_melhora` |
| aceitação `(delta, iteração) → bool` | `aceita_melhora`, `metropolis(temperaturas(...), passo)` | (as mesmas) |

Abaixo, cada peça aplicada uma vez na instância correlacionada.

In [ ]:
random.seed(1)
inicial = binaria.construtiva_aleatoria(itens, capacidade)
gulosa = binaria.construtiva_gulosa(itens, capacidade)
print("construtiva aleatória:", binaria.avaliar(inicial, itens, capacidade))
print("construtiva gulosa:   ", binaria.avaliar(gulosa, itens, capacidade))

vizinho = binaria.gerar_vizinho(inicial, itens, capacidade, k=1)
perturbada = binaria.perturbar(inicial, itens, capacidade, k=4)
print("movimento (1 bit):    ", binaria.avaliar(vizinho, itens, capacidade))
print("perturbação (k=4):    ", binaria.avaliar(perturbada, itens, capacidade))

melhor, n = binaria.busca_local_melhor_melhora(inicial, itens, capacidade)
primeira, m = binaria.busca_local_primeira_melhora(inicial, itens, capacidade)
print(f"busca local melhor melhora:   {binaria.avaliar(melhor, itens, capacidade)} ({n} avaliações)")
print(f"busca local primeira melhora: {binaria.avaliar(primeira, itens, capacidade)} ({m} avaliações)")

inicial_ordem = ordem.construtiva_aleatoria(itens, capacidade)
print("ordem aleatória:", ordem.avaliar(inicial_ordem, itens, capacidade),
      "· ordem gulosa:", ordem.avaliar(ordem.construtiva_gulosa(itens, capacidade), itens, capacidade))
otima_local, k = ordem.busca_local_primeira_melhora(inicial_ordem, itens, capacidade)
print(f"busca local por ordem: {ordem.avaliar(otima_local, itens, capacidade)} ({k} avaliações)")

aceitar = metropolis(temperaturas(1000, 1, 0.9), 100)
print("Metropolis, piora de 50 na iteração 1 (T = 1000):", aceitar(-50, 1), aceitar.ultimo)
print("só melhora:", aceita_melhora(-50, 1))

## 4. Debug: uma figura por etapa

O `Depurador` desenha o que o algoritmo enxerga em cada etapa: a solução atual (faixa no topo), o plano peso × valor com a trilha e as vizinhas, e o mini-histórico na mesma escala de valor.

- `show=True`: mostra as figuras aqui, abaixo da célula. `save=True`: grava em `plots/debug/<data-hora>/`.
- `a_cada=N`: desenha a cada N iterações (mais a primeira e a última).
- `limite`: número máximo de figuras. Um SA de 6.600 iterações com `a_cada=1` geraria 6.600 figuras.
- Na representação por ordem, passe `decodificar=ordem.decodificar`.

O depurador não sorteia números, então o resultado com e sem debug é o mesmo.

In [ ]:
random.seed(3)
itens_pequenos = gerar_instancia_aleatoria(8, 20, 100)
capacidade_pequena = sum(item.peso for item in itens_pequenos) // 2
otimo_pequeno, _ = programacao_dinamica(itens_pequenos, capacidade_pequena)

debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, a_cada=10, limite=4)
random.seed(7)
_ = simulated_annealing_mochila(itens_pequenos, capacidade_pequena, 50, 1, 0.5, 5,
                                debug=debug)

In [ ]:
# no ILS, as figuras incluem cada passada da busca local e a perturbação
debug = Depurador(itens_pequenos, capacidade_pequena, valor_otimo=otimo_pequeno,
                  show=True, limite=4)
random.seed(1)
_ = iterated_local_search_mochila(itens_pequenos, capacidade_pequena, 2, 3, debug=debug)

## 5. Linha de base: SA, ILS e VNS puros

Cada esquema roda com as peças padrão (binárias) em várias sementes, na instância correlacionada.

Compare pelas **avaliações**, não pelas iterações: uma iteração do SA avalia 1 vizinho, enquanto uma do ILS ou do VNS roda uma busca local inteira, com centenas de avaliações.

In [ ]:
SEMENTES = range(10)
PARAMETROS_SA = dict(temperatura_inicial=1000, temperatura_minima=1,
                     fator_resfriamento=0.9, iteracoes_por_temperatura=100)

execucoes = {
    "SA": rodar_sementes(simulated_annealing_mochila, itens, capacidade, SEMENTES,
                         **PARAMETROS_SA),
    "ILS": rodar_sementes(iterated_local_search_mochila, itens, capacidade, SEMENTES,
                          max_iteracoes=250, intensidade_perturbacao=4),
    "VNS": rodar_sementes(variable_neighborhood_search_mochila, itens, capacidade,
                          SEMENTES, max_iteracoes=250, k_max=4),
}
linha_de_base = {rotulo: resumir(e, otimo) for rotulo, e in execucoes.items()}
print(tabela(linha_de_base))

In [ ]:
_ = plotar_comparacao({rotulo: e[0]["historico"] for rotulo, e in execucoes.items()},
                      valor_otimo=otimo, show=True)
_ = plotar_historico(execucoes["SA"][0]["historico"], valor_otimo=otimo,
                     eixo_x="iteracao", titulo="SA, semente 0", show=True)

## 6. Suas misturas

Para montar uma combinação:

1. escolha o esquema (o esqueleto): `simulated_annealing_mochila`, `iterated_local_search_mochila` ou `variable_neighborhood_search_mochila`;
2. troque as peças passando-as como parâmetro (`avaliar`, `construtiva`, `movimento` ou `perturbacao`, `busca_local`, `aceitacao`). Todas as peças de uma execução precisam ser da mesma representação, com o mesmo prefixo (`binaria.` ou `ordem.`). Não crie uma variável chamada `ordem`: ela esconderia o módulo;
3. rode em várias sementes e compare com a linha de base na tabela e no gráfico.

O molde abaixo traz, comentado, como trocar a representação do SA para "por ordem". Na representação por ordem com busca local, prefira instâncias de até ~50 itens: cada vizinho é decodificado do zero, e com 100 itens uma busca local leva perto de 1 s.

In [ ]:
minhas_misturas = {
    # "SA por ordem": (simulated_annealing_mochila, dict(
    #     **PARAMETROS_SA,
    #     avaliar=ordem.avaliar, construtiva=ordem.construtiva_aleatoria, movimento=ordem.gerar_vizinho,
    # )),
}

resumos = dict(linha_de_base)
historicos = {rotulo: e[0]["historico"] for rotulo, e in execucoes.items()}
for rotulo, (esquema, parametros) in minhas_misturas.items():
    execucoes_mistura = rodar_sementes(esquema, itens, capacidade, SEMENTES, **parametros)
    resumos[rotulo] = resumir(execucoes_mistura, otimo)
    historicos[rotulo] = execucoes_mistura[0]["historico"]

print(tabela(resumos))
_ = plotar_comparacao(historicos, valor_otimo=otimo, show=True)